## Import & Config

In [7]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')

## Load data

In [8]:
# Kiểm tra thư mục dữ liệu
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Lỗi: Không tìm thấy thư mục dữ liệu tại {DATA_DIR}")

# Đọc các bảng dữ liệu
df_products = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
df_customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
df_geography = pd.read_csv(os.path.join(DATA_DIR, 'geography.csv'))
df_orders = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'))
df_order_items = pd.read_csv(os.path.join(DATA_DIR, 'order_items.csv'), low_memory=False)
df_payments = pd.read_csv(os.path.join(DATA_DIR, 'payments.csv'))
df_returns = pd.read_csv(os.path.join(DATA_DIR, 'returns.csv'))
df_sales = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'))
df_web_traffic = pd.read_csv(os.path.join(DATA_DIR, 'web_traffic.csv'))

## Câu hỏi trắc nghiệm

**Q1:** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)
<br> **Đáp án:** C: 180 ngày

In [9]:
df_order_complete = df_orders[df_orders['order_status'] == 'delivered'].copy()
df_order_complete['order_date'] = pd.to_datetime(df_order_complete['order_date'])
df_lich_su_mua = df_order_complete[['customer_id', 'order_date']].drop_duplicates().sort_values(by=['customer_id', 'order_date'])
df_lich_su_mua['ngay_khoang_cach'] = df_lich_su_mua.groupby('customer_id')['order_date'].diff().dt.days
print(f"Câu 1 (Trung vị khoảng cách): {df_lich_su_mua['ngay_khoang_cach'].median()} ngày")

Câu 1 (Trung vị khoảng cách): 178.0 ngày


**Q2:** Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?
<br> **Đáp án:** D: Standard

In [10]:
df_products['margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']
res2 = df_products.groupby('segment')['margin'].mean().idxmax()
print(f"Câu 2 (Segment lợi nhuận nhất): {res2}")

Câu 2 (Segment lợi nhuận nhất): Standard


**Q3:** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?
<br> **Đáp án:** B: wrong_size

In [11]:
df_streetwear = pd.merge(df_returns, df_products[df_products['category'] == 'Streetwear'], on='product_id')
res3 = df_streetwear['return_reason'].value_counts().idxmax()
print(f"Câu 3 (Lý do trả hàng Streetwear): {res3}")

Câu 3 (Lý do trả hàng Streetwear): wrong_size


**Q4:** Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?
<br> **Đáp án:** C: email_campaign

In [12]:
df_traffic = pd.read_csv(os.path.join(DATA_DIR, 'web_traffic.csv'))
res4 = df_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f"Câu 4 (Nguồn traffic ổn định nhất): {res4}")

Câu 4 (Nguồn traffic ổn định nhất): email_campaign


**Q5:** Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?
<br> **Đáp án:** C: 39%

In [13]:
ti_le_promo = (df_order_items['promo_id'].notna().sum() / len(df_order_items)) * 100
print(f"Câu 5 (Tỷ lệ khuyến mãi): {ti_le_promo:.0f}%")

Câu 5 (Tỷ lệ khuyến mãi): 39%


**Q6:** Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)
<br> **Đáp án:** A: 55+

In [14]:
df_cust = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
df_merge_6 = pd.merge(df_orders, df_cust, on='customer_id')
res6 = (df_merge_6.groupby('age_group')['order_id'].count() / df_cust.groupby('age_group')['customer_id'].count()).idxmax()
print(f"Câu 6 (Độ tuổi mua nhiều nhất): {res6}")

Câu 6 (Độ tuổi mua nhiều nhất): 55+


**Q7:** Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?
<br> **Đáp án:** C: East

In [15]:
df_order_items['doanh_thu'] = df_order_items['quantity'] * df_order_items['unit_price']
df_merge_7 = pd.merge(df_order_items, df_order_complete, on='order_id')
df_merge_7 = pd.merge(df_merge_7, df_geography, on='zip')
res7 = df_merge_7.groupby('region')['doanh_thu'].sum().idxmax()
print(f"Câu 7 (Vùng doanh thu cao nhất): {res7}")

Câu 7 (Vùng doanh thu cao nhất): East


**Q8:** Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?
<br> **Đáp án:** A: credit_card

In [16]:
res8 = df_orders[df_orders['order_status'] == 'cancelled']['payment_method'].value_counts().idxmax()
print(f"Câu 8 (Thanh toán đơn hủy nhiều nhất): {res8}")

Câu 8 (Thanh toán đơn hủy nhiều nhất): credit_card


**Q9:** Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?
<br> **Đáp án:** A: S

In [17]:
df_item_size = pd.merge(df_order_items, df_products[['product_id', 'size']], on='product_id')
df_ret_size = pd.merge(df_returns, df_products[['product_id', 'size']], on='product_id')
res9 = (df_ret_size['size'].value_counts() / df_item_size['size'].value_counts()).idxmax()
print(f"Câu 9 (Size có tỷ lệ trả hàng cao nhất): {res9}")

Câu 9 (Size có tỷ lệ trả hàng cao nhất): S


**Q10:** Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?
<br> **Đáp án:** C: 6 kỳ

In [18]:
df_q10 = df_payments.groupby('installments')['payment_value'].mean()
res10 = df_q10.idxmax()
print(f"Câu 10 (Gói trả góp cao nhất): {res10} kỳ")

Câu 10 (Gói trả góp cao nhất): 6 kỳ
